# Track 2: Regime of the day (Eshrit)

**Hypothesis.** The incumbent ridge applies one correction rule to every day. But the 15:30 surprise means
different things on different days. If the whole market is running hot because of a macro shock, a stock's high
volume is mostly the market, and the auction tends to follow only part of it. If one stock is hot on a quiet day,
that is stock-specific news, and the auction may respond differently. On index rebalance days, the auction is
set by index flows that intraday volume only partly reveals. The live 15:30 cross-section can tell these days
apart: market-wide pace, last-30-minute pace, size of price moves, and how dispersed the surprises are across stocks.

**Expected effect.** Biggest gains on event and rebalance days (Russell, MSCI, quarterly expiry) and on high-stress
days. Little change on ordinary days.

**What I try (four ridge variants, same 73 incumbent features plus the additions):**

| Variant | Added features | Why |
|---|---|---|
| `track2_events_only_v1` | Three rebalance-day flags only (Russell, MSCI, S&P quarterly) | Ablation: how much comes from scheduled events alone? |
| `track2_market_state_v1` | Six raw 15:30 market-state numbers + three rebalance-day flags | Ablation: does market state help without any clustering? |
| `track2_gmm_probs_v1` | GMM regime probabilities (K = 2, 3 or 4, chosen on validation) + rebalance flags | Does a compact regime label carry the signal? |
| `track2_gmm_interact_v1` | Regime probabilities × the stock's log volume surprise + rebalance flags | Lets the strength of the correction differ by regime (a soft version of one ridge per regime) |

**Success.** The gate. Otherwise, a clear improvement on the event-day segments without hurting the total.

**Leakage controls.**
- Market state uses only 15:30 values from the full, unsampled panel.
- The GMM and its scaling are fitted only on dates before the rows they label. Tune and validation rows use a GMM
  fitted before the validation window; refit and test rows use one fitted before the test year.
- K and the ridge strength are chosen on `fold.valid` only.

In [1]:
import sys
from pathlib import Path
from functools import lru_cache
import numpy as np, pandas as pd
from sklearn.mixture import GaussianMixture
sys.path.insert(0, str(Path.cwd()))
from intraday_harness import Harness, leaderboard, RIDGE_GRID, REBALANCE, SEED
from notebooks.modeling_utils import fit_ridge, predict_ridge

OWNER = "Eshrit"
QUICK = False                     # True = 2024 only while iterating; set False for the full 2022-2025 run
K_GRID = (2, 3, 4)                # number of regimes, chosen on the validation year
h = Harness(quick=QUICK)

## 1. Market state at 15:30, one row per date

Computed from every stock with a valid 15:30 checkpoint on that date (the full panel, not the training sample).
`cp_volume_surprise20` is the stock's volume so far divided by its 20-day average at 15:30.

In [2]:
cols = ["date", "permno", "cp_available", "cp_volume_surprise20", "cp_trade_count_surprise20",
        "cp_recent_30m_volume_share", "cp_vwap_return_prior_close"]
panel = h.full_panel("2021-01-01", "2026-01-01", cols)
live = panel[panel.cp_available.astype(bool)].copy()
live["log_surprise"] = np.log(live.cp_volume_surprise20.clip(lower=1e-3))
live["log_trade_surprise"] = np.log(live.cp_trade_count_surprise20.clip(lower=1e-3))
live["abs_move"] = live.cp_vwap_return_prior_close.abs()
live["hot"] = live.log_surprise > np.log(1.5)

g = live.groupby("date")
state = pd.DataFrame({
    "mkt_pace": g.log_surprise.median(),                       # how busy the market is versus normal
    "mkt_trade_pace": g.log_trade_surprise.median(),
    "mkt_dispersion": g.log_surprise.quantile(0.75) - g.log_surprise.quantile(0.25),  # stock-specific vs common
    "mkt_breadth": g.hot.mean(),                               # share of stocks running 50%+ above normal
    "mkt_recent30": g.cp_recent_30m_volume_share.median(),     # late-day acceleration
    "mkt_abs_move": g.abs_move.median(),                       # size of price moves so far today
})
STATE_COLS = list(state.columns)

# Scheduled index events, known in advance, aggregated to the date.
reb = h.con.sql(f'''SELECT CAST(date AS DATE) AS date, max(russell_recon) AS mkt_russell_recon,
                           max(msci_review) AS mkt_msci_review, max(sp_quarterly_rebal) AS mkt_sp_rebal
                    FROM read_parquet('{REBALANCE}') GROUP BY 1''').df()
reb["date"] = pd.to_datetime(reb.date)
state = state.join(reb.set_index("date")).fillna({"mkt_russell_recon": 0, "mkt_msci_review": 0, "mkt_sp_rebal": 0})
EVENT_COLS = ["mkt_russell_recon", "mkt_msci_review", "mkt_sp_rebal"]
print(f"{len(state)} dates, {state.index.min():%Y-%m-%d} to {state.index.max():%Y-%m-%d}")
state.describe().T.round(3)

1245 dates, 2021-01-04 to 2025-12-31


,count,mean,std,min,25%,50%,75%,max
mkt_pace,1245.0,-0.134,0.160,-0.706,-0.226,-0.136,-0.047,0.725
mkt_trade_pace,1245.0,-0.070,0.137,-0.536,-0.159,-0.074,0.004,0.623
mkt_dispersion,1245.0,0.552,0.047,0.429,0.520,0.547,0.578,0.769
mkt_breadth,1245.0,0.133,0.074,0.013,0.088,0.120,0.158,0.764
mkt_recent30,1245.0,0.099,0.009,0.078,0.094,0.098,0.104,0.157
mkt_abs_move,1245.0,0.011,0.005,0.004,0.008,0.010,0.012,0.057
mkt_russell_recon,1245.0,0.004,0.063,0.000,0.000,0.000,0.000,1.000
mkt_msci_review,1245.0,0.016,0.126,0.000,0.000,0.000,0.000,1.000
mkt_sp_rebal,1245.0,0.016,0.126,0.000,0.000,0.000,0.000,1.000


## 2. Regimes: a Gaussian mixture on the six state numbers

Fitted on dates before a cut-off only. Components are re-ordered by market pace (p0 = quietest) so the labels read
the same way across folds; the ridge refits per fold, so the order does not affect the forecast.

In [3]:
@lru_cache(maxsize=None)
def regimes(train_end: str, k: int) -> pd.DataFrame:
    train = state.loc[state.index < pd.Timestamp(train_end), STATE_COLS]
    mu, sd = train.mean(), train.std()
    gmm = GaussianMixture(k, covariance_type="full", reg_covar=1e-3, n_init=5, random_state=SEED)
    gmm.fit(((train - mu) / sd).to_numpy())
    p = gmm.predict_proba(((state[STATE_COLS] - mu) / sd).to_numpy())
    order = np.argsort(gmm.means_[:, STATE_COLS.index("mkt_pace")])
    return pd.DataFrame(p[:, order], index=state.index, columns=[f"regime{k}_p{j}" for j in range(k)])

def valid_start(year):
    return "2021-10-01" if year == 2022 else f"{year - 1}-01-01"

def regime_features(frame, fold_year, role):
    # Tune/valid rows are labelled by a GMM fitted before the validation window, refit/test rows by one
    # fitted before the test year, so nothing is fitted on the rows it labels or on later dates.
    train_end = valid_start(fold_year) if role in ("tune", "valid") else f"{fold_year}-01-01"
    out = state[STATE_COLS + EVENT_COLS].reindex(frame.date).set_axis(frame.index)
    log_surprise = np.log(frame.cp_volume_surprise20.clip(lower=1e-3))
    for k in K_GRID:
        p = regimes(train_end, k).reindex(frame.date).set_axis(frame.index)
        out = out.join(p)
        for c in p:
            out[f"{c}_x_surprise"] = p[c] * log_surprise
    return out

h.add_features(regime_features)

def prob_cols(k): return [f"regime{k}_p{j}" for j in range(k)]
def interact_cols(k): return [f"{c}_x_surprise" for c in prob_cols(k)]

VARIANTS = {   # name -> {K: extra columns}; K=None means no GMM
    "track2_events_only_v1": {None: EVENT_COLS},
    "track2_market_state_v1": {None: STATE_COLS + EVENT_COLS},
    "track2_gmm_probs_v1": {k: prob_cols(k) + EVENT_COLS for k in K_GRID},
    "track2_gmm_interact_v1": {k: prob_cols(k) + interact_cols(k) + EVENT_COLS for k in K_GRID},
}
DESCRIPTIONS = {
    "track2_events_only_v1": "Ridge + market-wide rebalance-day flags (Russell, MSCI, S&P quarterly) only; ablation for track 2",
    "track2_market_state_v1": "Ridge + six 15:30 market-state features (pace, dispersion, breadth, late pace, moves) + rebalance-day flags",
    "track2_gmm_probs_v1": "Ridge + GMM regime probabilities on the 15:30 market state (K chosen on validation) + rebalance-day flags",
    "track2_gmm_interact_v1": "Ridge + GMM regime probabilities and probability x stock log surprise interactions + rebalance-day flags",
}

### What the regimes look like (K = 3, fitted on 2021–2023, for description only)

In [4]:
p = regimes("2024-01-01", 3)
desc = state[STATE_COLS + EVENT_COLS].assign(regime=p.to_numpy().argmax(1))
desc = desc[desc.index < "2024-01-01"]
summary = desc.groupby("regime").mean().round(3)
summary.insert(0, "days", desc.groupby("regime").size())
summary

,days,mkt_pace,mkt_trade_pace,mkt_dispersion,mkt_breadth,mkt_recent30,mkt_abs_move,mkt_russell_recon,mkt_msci_review,mkt_sp_rebal
regime,,,,,,,,,,
0,221,-0.275,-0.188,0.533,0.077,0.096,0.012,0.000,0.009,0.005
1,423,-0.118,-0.064,0.564,0.133,0.096,0.010,0.005,0.019,0.012
2,105,0.108,0.146,0.543,0.246,0.100,0.018,0.010,0.019,0.057


## 3. Fit, tune and submit

One pass over the folds fits all four variants, so each fold is loaded once. For each variant, K and the ridge
strength are chosen together on the validation year, then the model is refitted through the year before the test year.

In [5]:
def tune_and_fit(fold, options):
    z_tune, bounds = fold.target(fold.tune), fold.bounds(fold.tune)
    y_valid = fold.valid.log1p_auction_volume.to_numpy(float)
    best = None
    for k, extra in options.items():
        prep = fold.preprocessor(fold.tune, extra_continuous=extra)
        x_tune, x_valid = prep.transform(fold.tune), prep.transform(fold.valid)
        for strength in RIDGE_GRID:
            beta = fit_ridge(x_tune, z_tune, strength)
            forecast = fold.to_shares(predict_ridge(x_valid, beta), fold.valid, bounds)
            rmse = float(np.sqrt(np.mean((np.log1p(forecast) - y_valid) ** 2)))
            if best is None or rmse < best[0]:
                best = (rmse, k, strength)
    rmse, k, strength = best
    prep = fold.preprocessor(fold.refit, extra_continuous=options[k])
    beta = fit_ridge(prep.transform(fold.refit), fold.target(fold.refit), strength)
    return predict_ridge(prep.transform(fold.test), beta), {"k": k, "strength": strength, "valid_rmse": rmse}

chosen = []
for fold in h.folds():
    for name, options in VARIANTS.items():
        correction, settings = tune_and_fit(fold, options)
        h.submit(fold, correction, name=name, owner=OWNER, description=DESCRIPTIONS[name])
        chosen.append({"year": fold.year, "variant": name, **settings})
pd.DataFrame(chosen)

track2_events_only_v1 2022: log RMSE 0.49434, WAPE 34.9996%, bias -15.34%


track2_market_state_v1 2022: log RMSE 0.49691, WAPE 35.7139%, bias -9.32%


track2_gmm_probs_v1 2022: log RMSE 0.49374, WAPE 35.0083%, bias -14.88%


track2_gmm_interact_v1 2022: log RMSE 0.49369, WAPE 34.9998%, bias -14.87%


track2_events_only_v1 2023: log RMSE 0.44431, WAPE 33.4083%, bias -12.39%


track2_market_state_v1 2023: log RMSE 0.44449, WAPE 33.4587%, bias -11.80%


... (10 outputs left out)

track2_market_state_v1 2025: log RMSE 0.40514, WAPE 31.5675%, bias -5.11%


track2_gmm_probs_v1 2025: log RMSE 0.40402, WAPE 31.3873%, bias -5.67%


track2_gmm_interact_v1 2025: log RMSE 0.40403, WAPE 31.3948%, bias -5.67%


,year,variant,k,strength,valid_rmse
0,2022,track2_events_only_v1,NaN,0.0100,0.554785
1,2022,track2_market_state_v1,NaN,0.0010,0.552786
2,2022,track2_gmm_probs_v1,4.0,0.0100,0.553647
3,2022,track2_gmm_interact_v1,4.0,0.0100,0.553699
4,2023,track2_events_only_v1,NaN,0.0100,0.494362
5,2023,track2_market_state_v1,NaN,0.1000,0.493992
6,2023,track2_gmm_probs_v1,2.0,0.0100,0.493554
7,2023,track2_gmm_interact_v1,2.0,0.0100,0.493548
8,2024,track2_events_only_v1,NaN,0.0100,0.444967
9,2024,track2_market_state_v1,NaN,0.0100,0.443665


## 4. Results against the incumbent

In [6]:
results = {name: h.report(name) for name in VARIANTS}   # quick runs are not written to the leaderboard
pd.concat({n: r["summary"] for n, r in results.items()}).round(5)

development_rmse_log  \
                       model                                          
track2_events_only_v1  daily_anchor                         0.46403   
                       incumbent_tree_ridge                 0.45445   
                       track2_events_only_v1                0.45211   
track2_market_state_v1 daily_anchor                         0.46403   
                       incumbent_tree_ridge                 0.45445   
                       track2_market_state_v1               0.45296   
track2_gmm_probs_v1    daily_anchor                         0.46403   
                       incumbent_tree_ridge                 0.45445   
                       track2_gmm_probs_v1                  0.45160   
track2_gmm_interact_v1 daily_anchor                         0.46403   
                       incumbent_tree_ridge                 0.45445   
                       track2_gmm_interact_v1               0.45160   

                                               development_wape  \
                       model                                      
track2_events_only_v1  daily_anchor                     0.34862   
                       incumbent_tree_ridge             0.33838   
                       track2_events_only_v1            0.33641   
track2_market_state_v1 daily_anchor                     0.34862   
                       incumbent_tree_ridge             0.33838   
                       track2_market_state_v1           0.33912   
track2_gmm_probs_v1    daily_anchor                     0.34862   
                       incumbent_tree_ridge             0.33838   
                       track2_gmm_probs_v1              0.33615   
track2_gmm_interact_v1 daily_anchor                     0.34862   
                       incumbent_tree_ridge             0.33838   
                       track2_gmm_interact_v1           0.33608   

                                               development_bias  \
                       model                                      
track2_events_only_v1  daily_anchor                    -0.15592   
                       incumbent_tree_ridge            -0.13937   
                       track2_events_only_v1           -0.13568   
track2_market_state_v1 daily_anchor                    -0.15592   
                       incumbent_tree_ridge            -0.13937   
                       track2_market_state_v1          -0.11253   
track2_gmm_probs_v1    daily_anchor                    -0.15592   
                       incumbent_tree_ridge            -0.13937   
                       track2_gmm_probs_v1             -0.13515   
track2_gmm_interact_v1 daily_anchor                    -0.15592   
                       incumbent_tree_ridge            -0.13937   
                       track2_gmm_interact_v1          -0.13481   

                                               confirmation_rmse_log  \
                       model                                           
track2_events_only_v1  daily_anchor                          0.42314   
                       incumbent_tree_ridge                  0.40613   
                       track2_events_only_v1                 0.40397   
track2_market_state_v1 daily_anchor                          0.42314   
                       incumbent_tree_ridge                  0.40613   
                       track2_market_state_v1                0.40514   
track2_gmm_probs_v1    daily_anchor                          0.42314   
                       incumbent_tree_ridge                  0.40613   
                       track2_gmm_probs_v1                   0.40402   
track2_gmm_interact_v1 daily_anchor                          0.42314   
                       incumbent_tree_ridge                  0.40613   
                       track2_gmm_interact_v1                0.40403   

                                               confirmation_wape  
                       model                                      
track2_events_only_v1  daily_an

In [7]:
pd.concat({n: r["intervals"] for n, r in results.items()}).round(5)

period         metric  difference    lower  \
track2_events_only_v1  0   development       rmse_log    -0.00234 -0.00412   
                       1   development           wape    -0.00198 -0.00359   
                       2   development    signed_bias     0.00369  0.00050   
                       3   development  absolute_bias    -0.00369 -0.00652   
                       4  confirmation       rmse_log    -0.00216 -0.00682   
                       5  confirmation           wape    -0.00211 -0.00681   
                       6  confirmation    signed_bias     0.00213 -0.00447   
                       7  confirmation  absolute_bias    -0.00213 -0.00861   
track2_market_state_v1 0   development       rmse_log    -0.00149 -0.00366   
                       1   development           wape     0.00074 -0.00138   
                       2   development    signed_bias     0.02684  0.02029   
                       3   development  absolute_bias    -0.02684 -0.03283   
                       4  confirmation       rmse_log    -0.00100 -0.00592   
                       5  confirmation           wape    -0.00029 -0.00572   
                       6  confirmation    signed_bias     0.00815 -0.00061   
                       7  confirmation  absolute_bias    -0.00815 -0.01694   
track2_gmm_probs_v1    0   development       rmse_log    -0.00285 -0.00490   
                       1   development           wape    -0.00224 -0.00404   
                       2   development    signed_bias     0.00422  0.00087   
                       3   development  absolute_bias    -0.00422 -0.00709   
                       4  confirmation       rmse_log    -0.00211 -0.00665   
                       5  confirmation           wape    -0.00209 -0.00676   
                       6  confirmation    signed_bias     0.00257 -0.00461   
                       7  confirmation  absolute_bias    -0.00257 -0.00955   
track2_gmm_interact_v1 0   development       rmse_log    -0.00285 -0.00491   
                       1   development           wape    -0.00231 -0.00410   
                       2   development    signed_bias     0.00456  0.00118   
                       3   development  absolute_bias    -0.00456 -0.00750   
                       4  confirmation       rmse_log    -0.00211 -0.00666   
                       5  confirmation           wape    -0.00202 -0.00668   
                       6  confirmation    signed_bias     0.00249 -0.00462   
                       7  confirmation  absolute_bias    -0.00249 -0.00950   

                            upper  
track2_events_only_v1  0 -0.00087  
                       1 -0.00058  
                       2  0.00652  
                       3 -0.00050  
                       4  0.00114  
                       5  0.00163  
                       6  0.00861  
                       7  0.00447  
track2_market_state_v1 0  0.00035  
                       1  0.00254  
                       2  0.03283  
                       3 -0.02029  
                       4  0.00295  
                       5  0.00462  
                       6  0.01694  
                       7  0.00061  
track2_gmm_probs_v1    0 -0.00114  
                       1 -0.00063  
                       2  0.00709  
                       3 -0.00087  
                       4  0.00115  
                       5  0.00172  
                       6  0.00955  
                       7  0.00461  
track2_gmm_interact_v1 0 -0.00116  
                       1 -0.00071  
                       2  0.00750  
                       3 -0.00118  
                       4  0.00116  
                       5  0.00178  
                       6  0.00950  
                       7  0.00462

In [8]:
pd.DataFrame({n: r["gate"] for n, r in results.items()}) if not QUICK else "Gate is only applied on the full 2022-2025 run."

,track2_events_only_v1,track2_market_state_v1,track2_gmm_probs_v1,track2_gmm_interact_v1
dev_rmse_gain,0.002341,0.001493,0.002848,0.00285
gain_ok,True,False,True,True
worst_year_change,-0.000842,0.001723,-0.001449,-0.001496
annual_ok,True,True,True,True
wape_ok,True,False,True,True
interval_ok,True,False,True,True
passes_gate,True,False,True,True


Segment log RMSE (lower is better). The event-day rows are the ones this track is aimed at.

In [9]:
base = results[next(iter(VARIANTS))]["segments"][["daily_anchor", "incumbent_tree_ridge"]]
table = base.join(pd.DataFrame({n: r["segments"][n] for n, r in results.items()}))
table.join((table[list(VARIANTS)].sub(table.incumbent_tree_ridge, axis=0)).add_suffix(" vs incumbent")).round(4)

,daily_anchor,incumbent_tree_ridge,track2_events_only_v1,track2_market_state_v1,track2_gmm_probs_v1,track2_gmm_interact_v1,track2_events_only_v1 vs incumbent,track2_market_state_v1 vs incumbent,track2_gmm_probs_v1 vs incumbent,track2_gmm_interact_v1 vs incumbent
segment,,,,,,,,,,
all,0.4656,0.4561,0.4539,0.4548,0.4533,0.4533,-0.0022,-0.0013,-0.0027,-0.0027
fomc_day,0.4957,0.4796,0.4747,0.4871,0.4754,0.4762,-0.0049,0.0075,-0.0042,-0.0034
largest_1pct_auctions (diagnostic),1.1105,1.0681,1.0550,1.0361,1.0552,1.0556,-0.0132,-0.0320,-0.0129,-0.0125
low_liquidity,0.5213,0.5234,0.5218,0.5222,0.5209,0.5210,-0.0016,-0.0013,-0.0025,-0.0024
month_end,0.6563,0.6526,0.6368,0.6373,0.6363,0.6368,-0.0158,-0.0153,-0.0162,-0.0158
msci_review,0.7786,0.7734,0.7155,0.7140,0.7149,0.7149,-0.0579,-0.0594,-0.0586,-0.0586
quarterly_expiry,0.7610,0.7308,0.7296,0.7380,0.7314,0.7316,-0.0012,0.0072,0.0006,0.0008
russell_recon,1.0273,1.0112,1.0264,1.0041,1.0194,1.0210,0.0152,-0.0071,0.0083,0.0098
short_history,1.0171,0.9970,0.9962,0.9887,0.9956,0.9976,-0.0009,-0.0083,-0.0015,0.0006


In [10]:
leaderboard()

,timestamp_utc,name,owner,description,git_commit,development_rmse_log,development_wape,development_bias,confirmation_rmse_log,confirmation_wape,incumbent_development_rmse_log,incumbent_development_wape,dev_rmse_gain,gain_ok,worst_year_change,annual_ok,wape_ok,interval_ok,passes_gate
0,2026-11-26T16:10:42+00:00,track2_gmm_interact_v1,Eshrit,Ridge + GMM regime probabilities and probabili...,beb57a9,0.451602,0.336080,-0.134809,0.404027,0.313948,0.454451,0.338385,0.002850,True,-0.001496,True,True,True,True
1,2026-11-26T16:10:39+00:00,track2_gmm_probs_v1,Eshrit,Ridge + GMM regime probabilities on the 15:30 ...,beb57a9,0.451603,0.336145,-0.135150,0.404022,0.313873,0.454451,0.338385,0.002848,True,-0.001449,True,True,True,True
2,2026-11-26T16:10:33+00:00,track2_events_only_v1,Eshrit,Ridge + market-wide rebalance-day flags (Russe...,beb57a9,0.452110,0.336405,-0.135679,0.403971,0.313854,0.454451,0.338385,0.002341,True,-0.000842,True,True,True,True
3,2026-11-26T16:10:37+00:00,track2_market_state_v1,Eshrit,"Ridge + six 15:30 market-state features (pace,...",beb57a9,0.452958,0.339121,-0.112530,0.405135,0.315675,0.454451,0.338385,0.001493,False,0.001723,True,False,False,False


## Findings (full run, 2022–2025)

| Variant | Dev log RMSE | Gain vs incumbent | Dev WAPE | 2025 log RMSE | Gate |
|---|---|---|---|---|---|
| Incumbent | 0.45445 | | 33.84% | 0.40614 | |
| `track2_events_only_v1` | 0.45211 | 0.00234 | 33.64% | 0.40397 | pass |
| `track2_market_state_v1` | 0.45296 | 0.00149 | 33.91% | 0.40514 | fail (gain, WAPE, interval) |
| `track2_gmm_probs_v1` | **0.45160** | **0.00285** | 33.61% | 0.40402 | pass |
| `track2_gmm_interact_v1` | 0.45160 | 0.00285 | 33.61% | 0.40403 | pass |

1. **Most of the gain comes from knowing it is an index-event day.** Three market-wide flags (Russell, MSCI,
   S&P quarterly rebalance) pass the gate on their own. MSCI-review days improve by 0.058, and month-end days
   by 0.016; most MSCI reviews take effect at a month-end.
2. **GMM regimes add a small extra gain on top: 0.0005 dev log RMSE versus events only.** Its 95% interval
   is −0.00099 to −0.00002, so the gain is real but barely clears zero. It does not show up in 2025 (+0.00005).
   Raw market-state numbers without clustering are worse than the flags alone: they overfit and push bias.
3. **Regime-specific strength (interactions) adds nothing** beyond the probabilities.
4. **Russell reconstitution day gets worse** (+0.008 to +0.015). There is only one such day a year, so the
   flag is learned from very few training days.

**Best candidate:** `track2_gmm_probs_v1`. When presenting it, say that most of its gain comes from the event flags.

**Ideas for day 2:**
- Stock-level rebalance features (ETF ownership × event day) instead of market-wide flags, which might fix Russell.
- Track 3's calendar-adjusted surprise as a GMM input.
- Hand `regime*_p*` and the event flags to track 1's nonlinear model.
